# ST3 — Catalogue Intelligence

## Analyse produits, saisonnalité et pricing dynamique

Ce notebook correspond au sous-thème **ST3** du projet e-commerce.

L'objectif est simple : comprendre quels produits génèrent le plus de valeur, quels produits sont saisonniers, quels produits doivent être mis en avant, et quels produits peuvent faire l'objet d'une action de pricing.

À la fin, le notebook exporte des fichiers propres dans `data/gold/` pour préparer le dashboard L8.

## 1. Objectifs métier du notebook

Dans ce notebook, je cherche à répondre à plusieurs questions concrètes :

1. Quels sont les produits qui génèrent le plus de chiffre d'affaires ?  
2. Est-ce que le chiffre d'affaires est concentré sur une petite partie du catalogue ?  
3. Quels produits sont stratégiques selon la matrice BCG ?  
4. Quels mois ou semaines sont les plus forts en ventes ?  
5. Quels produits sont sensibles au prix ?  
6. Quelles actions pricing peut-on recommander ?

Le notebook mélange donc **code**, **graphiques Plotly** et **interprétations métier**.

## 2. Import des librairies

On importe les librairies nécessaires pour manipuler les données, créer les graphiques et exporter les résultats.

In [ ]:

# Manipulation des données
import pandas as pd
import numpy as np

# Visualisation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Chemins de fichiers
from pathlib import Path

# Affichage propre dans le notebook
from IPython.display import display, Markdown

# Options d'affichage
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 3. Détection automatique du dossier projet

Le notebook est prévu pour être placé dans le dossier `notebooks/`.  
Le code ci-dessous retrouve automatiquement la racine du projet, puis prépare les chemins vers :

- `data/silver/` : données propres en entrée ;
- `data/gold/` : fichiers exportés pour Power BI / Tableau.

In [ ]:

def find_project_root(start_path: Path = Path.cwd()) -> Path:
    """Retourne le dossier racine du projet en cherchant data/silver."""
    start_path = start_path.resolve()
    candidates = [start_path] + list(start_path.parents)

    for candidate in candidates:
        if (candidate / "data" / "silver").exists():
            return candidate

    raise FileNotFoundError(
        "Impossible de trouver le dossier data/silver. "
        "Place ce notebook dans le projet ou dans le dossier notebooks/."
    )

PROJECT_ROOT = find_project_root()
SILVER_DIR = PROJECT_ROOT / "data" / "silver"
GOLD_DIR = PROJECT_ROOT / "data" / "gold"

GOLD_DIR.mkdir(parents=True, exist_ok=True)

print("Racine projet :", PROJECT_ROOT)
print("Dossier Silver :", SILVER_DIR)
print("Dossier Gold :", GOLD_DIR)

## 4. Chargement des datasets ST3

Pour ST3, on utilise principalement :

- `online_retail_full.csv` : ventes, produits, quantités, prix, dates, pays ;
- `online_retail_returns.csv` : retours clients, utiles pour calculer le taux de retour produit.

Le notebook utilise `online_retail_full.csv` si disponible. Sinon, il bascule vers un autre fichier retail propre.

In [ ]:

def first_existing_file(folder: Path, filenames: list[str]) -> Path:
    """Retourne le premier fichier existant dans une liste de noms possibles."""
    for name in filenames:
        path = folder / name
        if path.exists():
            return path
    raise FileNotFoundError(f"Aucun fichier trouvé parmi : {filenames}")

sales_file = first_existing_file(
    SILVER_DIR,
    ["online_retail_full.csv", "online_retail_clean.csv", "data_cleaned.csv"]
)

returns_file = first_existing_file(
    SILVER_DIR,
    ["online_retail_returns.csv"]
)

print("Fichier ventes utilisé :", sales_file.name)
print("Fichier retours utilisé :", returns_file.name)

# Lecture des fichiers
sales = pd.read_csv(
    sales_file,
    dtype={"InvoiceNo": "str", "StockCode": "str", "Description": "str", "Country": "str"},
    low_memory=False
)

returns = pd.read_csv(
    returns_file,
    dtype={"InvoiceNo": "str", "StockCode": "str", "Description": "str", "Country": "str"},
    low_memory=False
)

print("Ventes :", sales.shape)
print("Retours :", returns.shape)

display(sales.head())

## 5. Préparation des données

Avant l'analyse, on sécurise les types de colonnes :

- les dates sont converties au format date ;
- les quantités et prix sont convertis en valeurs numériques ;
- le chiffre d'affaires est recalculé si nécessaire.

Ensuite, on garde uniquement les lignes de ventes positives pour l'analyse catalogue.

In [ ]:

# Conversion des dates
sales["InvoiceDate"] = pd.to_datetime(sales["InvoiceDate"], errors="coerce")
returns["InvoiceDate"] = pd.to_datetime(returns["InvoiceDate"], errors="coerce")

# Conversion des colonnes numériques
for df in [sales, returns]:
    for col in ["Quantity", "UnitPrice", "CustomerID"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

# Harmonisation du chiffre d'affaires
if "TotalRevenue" not in sales.columns:
    if "TotalPrice" in sales.columns:
        sales["TotalRevenue"] = sales["TotalPrice"]
    else:
        sales["TotalRevenue"] = sales["Quantity"] * sales["UnitPrice"]

if "TotalRevenue" not in returns.columns:
    if "TotalPrice" in returns.columns:
        returns["TotalRevenue"] = returns["TotalPrice"]
    else:
        returns["TotalRevenue"] = returns["Quantity"] * returns["UnitPrice"]

sales["TotalRevenue"] = pd.to_numeric(sales["TotalRevenue"], errors="coerce")
returns["TotalRevenue"] = pd.to_numeric(returns["TotalRevenue"], errors="coerce")

# Codes non produits à exclure de l'analyse catalogue
SPECIAL_CODES = {"POST", "D", "C2", "M", "BANK CHARGES", "DOT", "AMAZONFEE", "CRUK"}

sales_st3 = sales.copy()

# On garde les vraies ventes : quantité, prix et CA positifs
sales_st3 = sales_st3[
    (sales_st3["Quantity"] > 0) &
    (sales_st3["UnitPrice"] > 0) &
    (sales_st3["TotalRevenue"] > 0)
].copy()

# Exclusion des codes techniques / frais / remises
sales_st3 = sales_st3[~sales_st3["StockCode"].astype(str).str.upper().isin(SPECIAL_CODES)].copy()

# Suppression des lignes sans informations essentielles
sales_st3 = sales_st3.dropna(subset=["StockCode", "Description", "InvoiceDate"])

print("Lignes initiales ventes :", len(sales))
print("Lignes conservées pour ST3 :", len(sales_st3))
print("Produits uniques conservés :", sales_st3["StockCode"].nunique())

## 6. Contrôle rapide des données

Cette étape permet de vérifier la période couverte, le nombre de produits et le volume de ventes analysé.

In [ ]:

overview = pd.DataFrame({
    "Indicateur": [
        "Date début",
        "Date fin",
        "Nombre de lignes ventes ST3",
        "Nombre de produits uniques",
        "Nombre de commandes",
        "Nombre de clients identifiés",
        "Pays couverts"
    ],
    "Valeur": [
        sales_st3["InvoiceDate"].min(),
        sales_st3["InvoiceDate"].max(),
        len(sales_st3),
        sales_st3["StockCode"].nunique(),
        sales_st3["InvoiceNo"].nunique(),
        sales_st3["CustomerID"].nunique(),
        sales_st3["Country"].nunique()
    ]
})

display(overview)

## 7. KPIs catalogue

On calcule les premiers indicateurs clés du catalogue : chiffre d'affaires, quantité vendue, nombre de commandes, prix moyen et panier moyen.

In [ ]:

total_revenue = sales_st3["TotalRevenue"].sum()
total_quantity = sales_st3["Quantity"].sum()
nb_skus = sales_st3["StockCode"].nunique()
nb_orders = sales_st3["InvoiceNo"].nunique()
avg_unit_price = sales_st3["UnitPrice"].mean()
avg_order_value = sales_st3.groupby("InvoiceNo")["TotalRevenue"].sum().mean()

kpis = pd.DataFrame({
    "KPI": [
        "Chiffre d'affaires total",
        "Quantité vendue",
        "Nombre de SKUs",
        "Nombre de commandes",
        "Prix unitaire moyen",
        "Panier moyen"
    ],
    "Valeur": [
        total_revenue,
        total_quantity,
        nb_skus,
        nb_orders,
        avg_unit_price,
        avg_order_value
    ]
})

display(kpis)

### Lecture métier

Ces KPIs donnent une première vision du catalogue.  
Ils permettent de savoir si l'analyse porte sur un volume suffisant et de préparer les indicateurs utilisés dans le dashboard.

## 8. Construction de la table de performance produit

On crée une table à la maille **1 ligne = 1 produit / SKU**.

Pour chaque produit, on calcule :

- quantité vendue ;
- chiffre d'affaires total ;
- nombre de commandes ;
- nombre de clients ;
- prix moyen ;
- quantité retournée ;
- taux de retour ;
- part du chiffre d'affaires.

In [ ]:

# Agrégation des ventes par produit
product_sales = sales_st3.groupby("StockCode", as_index=False).agg(
    quantity_sold=("Quantity", "sum"),
    total_revenue=("TotalRevenue", "sum"),
    nb_orders=("InvoiceNo", "nunique"),
    nb_customers=("CustomerID", "nunique"),
    avg_unit_price=("UnitPrice", "mean"),
    first_sale=("InvoiceDate", "min"),
    last_sale=("InvoiceDate", "max")
)

# Description produit : on garde la dernière description disponible pour chaque SKU
product_desc = (
    sales_st3.sort_values("InvoiceDate")
    .drop_duplicates("StockCode", keep="last")[["StockCode", "Description"]]
    .rename(columns={"Description": "description"})
)

product_perf = product_sales.merge(product_desc, on="StockCode", how="left")

# Agrégation des retours par produit
returns_st3 = returns.copy()
returns_st3 = returns_st3[~returns_st3["StockCode"].astype(str).str.upper().isin(SPECIAL_CODES)].copy()

returns_agg = returns_st3.groupby("StockCode", as_index=False).agg(
    returned_qty=("Quantity", lambda x: x.abs().sum()),
    return_value=("TotalRevenue", lambda x: x.abs().sum()),
    nb_return_invoices=("InvoiceNo", "nunique")
)

# Fusion ventes + retours
product_perf = product_perf.merge(returns_agg, on="StockCode", how="left")

# Remplacer les retours manquants par 0
for col in ["returned_qty", "return_value", "nb_return_invoices"]:
    product_perf[col] = product_perf[col].fillna(0)

# Taux de retour en quantité
product_perf["return_rate_qty"] = (
    product_perf["returned_qty"] / product_perf["quantity_sold"]
).replace([np.inf, -np.inf], np.nan).fillna(0)

# Part de CA du produit dans le catalogue
product_perf["market_share_pct"] = product_perf["total_revenue"] / product_perf["total_revenue"].sum() * 100

# Tri par chiffre d'affaires
product_perf = product_perf.sort_values("total_revenue", ascending=False).reset_index(drop=True)
product_perf["rank_revenue"] = np.arange(1, len(product_perf) + 1)

# Affichage des premiers produits
cols_preview = [
    "StockCode", "description", "quantity_sold", "total_revenue", "nb_orders",
    "avg_unit_price", "returned_qty", "return_rate_qty", "market_share_pct"
]

display(product_perf[cols_preview].head(10))

## 9. Top 20 produits par chiffre d'affaires

Ce graphique permet d'identifier rapidement les produits qui contribuent le plus au chiffre d'affaires.

Ces produits doivent être surveillés en priorité : stock, prix, disponibilité, qualité et retours.

In [ ]:

top20_skus = product_perf.head(20).copy()
top20_skus["description_short"] = top20_skus["description"].str.slice(0, 55)

fig = px.bar(
    top20_skus.sort_values("total_revenue", ascending=True),
    x="total_revenue",
    y="description_short",
    orientation="h",
    title="Top 20 des produits par chiffre d'affaires",
    labels={"total_revenue": "Chiffre d'affaires", "description_short": "Produit"},
    hover_data=["StockCode", "quantity_sold", "nb_orders", "avg_unit_price", "return_rate_qty"]
)

fig.update_layout(height=650)
fig.show()

In [ ]:

best_product = top20_skus.iloc[0]

message = f"""
### Lecture métier — Top produits

Le produit qui génère le plus de chiffre d'affaires est **{best_product['description']}** (`{best_product['StockCode']}`).  
Il représente un chiffre d'affaires d'environ **{best_product['total_revenue']:,.0f}**.

Ces produits du Top 20 sont critiques pour le business : une rupture de stock ou une mauvaise décision prix sur ces références peut avoir un impact direct sur le chiffre d'affaires global.
"""

display(Markdown(message))

## 10. Analyse ABC / Pareto

L'analyse ABC sert à classer les produits selon leur contribution au chiffre d'affaires.

- **Classe A** : produits les plus importants, environ 80 % du CA ;
- **Classe B** : produits intermédiaires, environ 15 % du CA ;
- **Classe C** : produits à faible contribution, environ 5 % du CA.

Cette analyse aide à prioriser les efforts commerciaux et marketing.

In [ ]:

# Calcul du CA cumulé
product_perf["cumulative_revenue"] = product_perf["total_revenue"].cumsum()
product_perf["cumulative_revenue_pct"] = (
    product_perf["cumulative_revenue"] / product_perf["total_revenue"].sum() * 100
)

# Affectation des classes ABC
product_perf["abc_class"] = np.select(
    [
        product_perf["cumulative_revenue_pct"] <= 80,
        product_perf["cumulative_revenue_pct"] <= 95
    ],
    ["A", "B"],
    default="C"
)

abc_summary = product_perf.groupby("abc_class", as_index=False).agg(
    nb_skus=("StockCode", "nunique"),
    total_revenue=("total_revenue", "sum"),
    quantity_sold=("quantity_sold", "sum")
)

abc_summary["revenue_share_pct"] = abc_summary["total_revenue"] / abc_summary["total_revenue"].sum() * 100
abc_summary["sku_share_pct"] = abc_summary["nb_skus"] / abc_summary["nb_skus"].sum() * 100

# Ordre logique A, B, C
abc_summary["abc_class"] = pd.Categorical(abc_summary["abc_class"], categories=["A", "B", "C"], ordered=True)
abc_summary = abc_summary.sort_values("abc_class")

display(abc_summary)

In [ ]:

# Courbe Pareto sur les 300 premiers produits pour garder un graphique lisible
pareto_view = product_perf.head(300).copy()

fig = px.line(
    pareto_view,
    x="rank_revenue",
    y="cumulative_revenue_pct",
    title="Courbe Pareto — CA cumulé par produits",
    labels={"rank_revenue": "Rang du produit", "cumulative_revenue_pct": "CA cumulé (%)"},
    hover_data=["StockCode", "description", "total_revenue", "abc_class"]
)

fig.add_hline(y=80, line_dash="dash", annotation_text="Seuil 80 %")
fig.add_hline(y=95, line_dash="dash", annotation_text="Seuil 95 %")
fig.update_layout(height=500)
fig.show()

In [ ]:

nb_a = int(abc_summary.loc[abc_summary["abc_class"] == "A", "nb_skus"].iloc[0])
share_a = float(abc_summary.loc[abc_summary["abc_class"] == "A", "revenue_share_pct"].iloc[0])

message = f"""
### Lecture métier — ABC

La classe **A** contient **{nb_a} produits** et génère environ **{share_a:.1f} %** du chiffre d'affaires.  
Cela confirme que le catalogue n'a pas la même importance partout : certains produits doivent être pilotés de manière prioritaire.

Action recommandée : suivre les produits **A** avec des KPIs dédiés : disponibilité, prix, retours et saisonnalité.
"""

display(Markdown(message))

## 11. Matrice BCG du catalogue

La matrice BCG permet de classer les produits selon deux dimensions :

- **part de chiffre d'affaires** : poids du produit dans le catalogue ;
- **croissance** : évolution du CA entre la première et la deuxième moitié de la période.

Les quadrants sont :

- **Étoile** : produit fort et en croissance ;
- **Vache à lait** : produit fort mais croissance plus faible ;
- **Dilemme** : produit encore faible mais en croissance ;
- **Poids mort** : produit faible et peu dynamique.

In [ ]:

# Découpage de la période en deux parties
min_date = sales_st3["InvoiceDate"].min()
max_date = sales_st3["InvoiceDate"].max()
split_date = min_date + (max_date - min_date) / 2

sales_st3["period_bcg"] = np.where(
    sales_st3["InvoiceDate"] <= split_date,
    "first_period",
    "second_period"
)

# CA par produit et par période
bcg_pivot = (
    sales_st3.groupby(["StockCode", "period_bcg"])["TotalRevenue"]
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["first_period", "second_period"]:
    if col not in bcg_pivot.columns:
        bcg_pivot[col] = 0

# Croissance du CA entre les deux périodes
bcg_pivot["growth_pct"] = np.where(
    bcg_pivot["first_period"] > 0,
    (bcg_pivot["second_period"] - bcg_pivot["first_period"]) / bcg_pivot["first_period"] * 100,
    np.nan
)

product_perf = product_perf.merge(
    bcg_pivot[["StockCode", "first_period", "second_period", "growth_pct"]],
    on="StockCode",
    how="left"
)

# Seuils de classification
share_threshold = product_perf["market_share_pct"].mean()
growth_threshold = product_perf["growth_pct"].replace([np.inf, -np.inf], np.nan).median()

# Affectation des quadrants
conditions = [
    (product_perf["market_share_pct"] >= share_threshold) & (product_perf["growth_pct"] >= growth_threshold),
    (product_perf["market_share_pct"] >= share_threshold) & ~(product_perf["growth_pct"] >= growth_threshold),
    (product_perf["market_share_pct"] < share_threshold) & (product_perf["growth_pct"] >= growth_threshold),
]

choices = ["Étoile", "Vache à lait", "Dilemme"]
product_perf["bcg_quadrant"] = np.select(conditions, choices, default="Poids mort")

bcg_summary = product_perf.groupby("bcg_quadrant", as_index=False).agg(
    nb_skus=("StockCode", "nunique"),
    total_revenue=("total_revenue", "sum"),
    avg_growth_pct=("growth_pct", "mean")
)

bcg_summary["revenue_share_pct"] = bcg_summary["total_revenue"] / product_perf["total_revenue"].sum() * 100

display(bcg_summary.sort_values("total_revenue", ascending=False))

In [ ]:

# Pour éviter que quelques outliers écrasent le graphique, on affiche la croissance bornée.
bcg_plot = product_perf.sort_values("total_revenue", ascending=False).head(600).copy()
bcg_plot["growth_pct_plot"] = bcg_plot["growth_pct"].clip(lower=-100, upper=500)
bcg_plot["description_short"] = bcg_plot["description"].str.slice(0, 60)

fig = px.scatter(
    bcg_plot,
    x="market_share_pct",
    y="growth_pct_plot",
    size="total_revenue",
    color="bcg_quadrant",
    hover_name="description_short",
    hover_data=["StockCode", "abc_class", "total_revenue", "growth_pct", "market_share_pct"],
    title="Matrice BCG — Produits par part de CA et croissance",
    labels={
        "market_share_pct": "Part du CA (%)",
        "growth_pct_plot": "Croissance du CA (%)",
        "bcg_quadrant": "Quadrant BCG"
    }
)

fig.add_vline(x=share_threshold, line_dash="dash", annotation_text="Seuil part CA")
fig.add_hline(y=growth_threshold, line_dash="dash", annotation_text="Seuil croissance")
fig.update_layout(height=650)
fig.show()

In [ ]:

star_count = int((product_perf["bcg_quadrant"] == "Étoile").sum())
cashcow_count = int((product_perf["bcg_quadrant"] == "Vache à lait").sum())

message = f"""
### Lecture métier — BCG

La matrice BCG permet de distinguer les produits à fort potentiel des produits à faible contribution.  
Dans cette analyse, on identifie **{star_count} produits Étoiles** et **{cashcow_count} produits Vaches à lait**.

- Les **Étoiles** doivent être mises en avant et sécurisées en stock.  
- Les **Vaches à lait** doivent être maintenues, avec une optimisation progressive du prix.  
- Les **Dilemmes** méritent des tests de visibilité ou de promotion.  
- Les **Poids morts** peuvent être soldés, dépriorisés ou retirés du catalogue.
"""

display(Markdown(message))

## 12. Analyse de la saisonnalité

L'objectif est de comprendre à quel moment le catalogue vend le plus.

On analyse :

- le chiffre d'affaires par mois ;
- le chiffre d'affaires par semaine ;
- le chiffre d'affaires par jour de semaine ;
- une heatmap mois x jour de semaine.

In [ ]:

# Variables temporelles
sales_st3["month"] = sales_st3["InvoiceDate"].dt.to_period("M").astype(str)
sales_st3["week"] = sales_st3["InvoiceDate"].dt.to_period("W").apply(lambda r: r.start_time)
sales_st3["weekday"] = sales_st3["InvoiceDate"].dt.day_name()

weekday_map = {
    "Monday": "Lundi",
    "Tuesday": "Mardi",
    "Wednesday": "Mercredi",
    "Thursday": "Jeudi",
    "Friday": "Vendredi",
    "Saturday": "Samedi",
    "Sunday": "Dimanche"
}

sales_st3["weekday_fr"] = sales_st3["weekday"].map(weekday_map)

# CA mensuel
monthly_revenue = (
    sales_st3.groupby("month", as_index=False)["TotalRevenue"]
    .sum()
    .rename(columns={"TotalRevenue": "monthly_revenue"})
)

# CA hebdomadaire
weekly_revenue = (
    sales_st3.groupby("week", as_index=False)["TotalRevenue"]
    .sum()
    .rename(columns={"TotalRevenue": "weekly_revenue"})
)

# CA par jour de semaine
weekday_order = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
weekday_revenue = (
    sales_st3.groupby("weekday_fr", as_index=False)["TotalRevenue"]
    .sum()
    .rename(columns={"TotalRevenue": "weekday_revenue"})
)
weekday_revenue["weekday_fr"] = pd.Categorical(weekday_revenue["weekday_fr"], categories=weekday_order, ordered=True)
weekday_revenue = weekday_revenue.sort_values("weekday_fr")

print("CA mensuel :")
display(monthly_revenue)

In [ ]:

fig = px.line(
    monthly_revenue,
    x="month",
    y="monthly_revenue",
    markers=True,
    title="Évolution mensuelle du chiffre d'affaires",
    labels={"month": "Mois", "monthly_revenue": "Chiffre d'affaires"}
)

fig.update_layout(height=500)
fig.show()

In [ ]:

fig = px.line(
    weekly_revenue,
    x="week",
    y="weekly_revenue",
    title="Évolution hebdomadaire du chiffre d'affaires",
    labels={"week": "Semaine", "weekly_revenue": "Chiffre d'affaires"}
)

fig.update_layout(height=500)
fig.show()

In [ ]:

fig = px.bar(
    weekday_revenue,
    x="weekday_fr",
    y="weekday_revenue",
    title="Chiffre d'affaires par jour de semaine",
    labels={"weekday_fr": "Jour", "weekday_revenue": "Chiffre d'affaires"}
)

fig.update_layout(height=500)
fig.show()

In [ ]:

# Heatmap mois x jour de semaine
heatmap_data = (
    sales_st3.groupby(["month", "weekday_fr"], as_index=False)["TotalRevenue"]
    .sum()
    .rename(columns={"TotalRevenue": "revenue"})
)

heatmap_pivot = heatmap_data.pivot(index="month", columns="weekday_fr", values="revenue")
heatmap_pivot = heatmap_pivot.reindex(columns=weekday_order)

fig = px.imshow(
    heatmap_pivot,
    aspect="auto",
    title="Heatmap saisonnalité — CA par mois et jour de semaine",
    labels={"x": "Jour de semaine", "y": "Mois", "color": "CA"}
)

fig.update_layout(height=650)
fig.show()

In [ ]:

best_month_row = monthly_revenue.loc[monthly_revenue["monthly_revenue"].idxmax()]
weak_month_row = monthly_revenue.loc[monthly_revenue["monthly_revenue"].idxmin()]

message = f"""
### Lecture métier — Saisonnalité

Le mois le plus fort est **{best_month_row['month']}**, avec un chiffre d'affaires d'environ **{best_month_row['monthly_revenue']:,.0f}**.  
Le mois le plus faible est **{weak_month_row['month']}**, avec environ **{weak_month_row['monthly_revenue']:,.0f}**.

Cette information est utile pour planifier les stocks, les campagnes marketing et les promotions.  
Une promotion lancée au mauvais moment peut réduire la marge sans créer beaucoup de volume supplémentaire.
"""

display(Markdown(message))

## 13. Décomposition STL indicative

La décomposition STL sépare une série temporelle en trois éléments :

- **tendance** : orientation générale du chiffre d'affaires ;
- **saisonnalité** : cycles réguliers ;
- **résidu** : variations exceptionnelles.

Comme la base couvre environ 13 mois, cette décomposition reste **indicative**. Elle aide surtout à préparer la lecture dashboard.

In [ ]:

try:
    from statsmodels.tsa.seasonal import STL

    weekly_series = weekly_revenue.copy()
    weekly_series = weekly_series.set_index("week").sort_index()
    weekly_series = weekly_series.asfreq("W-MON")
    weekly_series["weekly_revenue"] = weekly_series["weekly_revenue"].fillna(0)

    # Période indicative de 13 semaines pour détecter des cycles infra-annuels
    stl = STL(weekly_series["weekly_revenue"], period=13, robust=True)
    result = stl.fit()

    stl_df = weekly_series.copy()
    stl_df["trend"] = result.trend
    stl_df["seasonal"] = result.seasonal
    stl_df["resid"] = result.resid

    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        subplot_titles=["CA hebdomadaire", "Tendance", "Saisonnalité", "Résidu"]
    )

    fig.add_trace(go.Scatter(x=stl_df.index, y=stl_df["weekly_revenue"], mode="lines", name="CA"), row=1, col=1)
    fig.add_trace(go.Scatter(x=stl_df.index, y=stl_df["trend"], mode="lines", name="Tendance"), row=2, col=1)
    fig.add_trace(go.Scatter(x=stl_df.index, y=stl_df["seasonal"], mode="lines", name="Saisonnalité"), row=3, col=1)
    fig.add_trace(go.Scatter(x=stl_df.index, y=stl_df["resid"], mode="lines", name="Résidu"), row=4, col=1)

    fig.update_layout(height=900, title="Décomposition STL indicative du CA hebdomadaire")
    fig.show()

except Exception as e:
    print("STL non exécutée. Raison :", e)
    print("Si nécessaire, installer statsmodels : pip install statsmodels")

## 14. Analyse de l'élasticité prix

L'élasticité prix mesure la sensibilité des quantités vendues à une variation de prix.

Formule utilisée :

$$Elasticité = \frac{Variation\ \%\ de\ la\ quantité}{Variation\ \%\ du\ prix}$$

Interprétation simple :

- entre **0 et -1** : produit plutôt inélastique, une hausse de prix modérée peut être testée ;
- inférieur à **-1** : produit élastique, attention aux hausses de prix ;
- supérieur à **0** : comportement atypique ou effet produit complémentaire.

Pour rester robuste, on calcule l'élasticité sur les **50 produits les plus vendus**.

In [ ]:

# Sélection des 50 produits les plus vendus
TOP_N_ELASTICITY = 50
top50_products = product_perf.sort_values("quantity_sold", ascending=False).head(TOP_N_ELASTICITY)["StockCode"]

# Agrégation mensuelle par produit
monthly_product = (
    sales_st3[sales_st3["StockCode"].isin(top50_products)]
    .groupby(["StockCode", "month"], as_index=False)
    .agg(
        monthly_qty=("Quantity", "sum"),
        monthly_revenue=("TotalRevenue", "sum")
    )
)

monthly_product["avg_monthly_price"] = monthly_product["monthly_revenue"] / monthly_product["monthly_qty"]
monthly_product = monthly_product.sort_values(["StockCode", "month"])

# Variations mois par mois
monthly_product["price_pct_change"] = monthly_product.groupby("StockCode")["avg_monthly_price"].pct_change()
monthly_product["qty_pct_change"] = monthly_product.groupby("StockCode")["monthly_qty"].pct_change()

# Calcul de l'élasticité observée
monthly_product["elasticity_obs"] = monthly_product["qty_pct_change"] / monthly_product["price_pct_change"]
monthly_product = monthly_product.replace([np.inf, -np.inf], np.nan)

# On garde les observations avec une vraie variation de prix et sans valeurs extrêmes
elasticity_observations = monthly_product[
    (monthly_product["price_pct_change"].abs() >= 0.02) &
    (monthly_product["elasticity_obs"].abs() < 20)
].copy()

# Élasticité médiane par produit
elasticity = (
    elasticity_observations.groupby("StockCode", as_index=False)
    .agg(
        elasticity=("elasticity_obs", "median"),
        nb_elasticity_points=("elasticity_obs", "count")
    )
)

# Fusion avec la table produit
product_perf = product_perf.merge(elasticity, on="StockCode", how="left")

# Profil d'élasticité
def classify_elasticity(value):
    if pd.isna(value):
        return "Non calculable"
    if -1 <= value < 0:
        return "Inélastique"
    if value < -1:
        return "Élastique"
    if value > 0:
        return "Atypique / complémentaire"
    return "Stable"

product_perf["elasticity_profile"] = product_perf["elasticity"].apply(classify_elasticity)

elasticity_preview = product_perf[
    ["StockCode", "description", "quantity_sold", "avg_unit_price", "elasticity", "elasticity_profile", "nb_elasticity_points"]
].sort_values("quantity_sold", ascending=False).head(20)

display(elasticity_preview)

In [ ]:

elasticity_plot = product_perf.dropna(subset=["elasticity"]).copy()
elasticity_plot = elasticity_plot.sort_values("elasticity").head(30)
elasticity_plot["description_short"] = elasticity_plot["description"].str.slice(0, 55)

fig = px.bar(
    elasticity_plot,
    x="elasticity",
    y="description_short",
    orientation="h",
    title="Produits les plus sensibles au prix — élasticité estimée",
    labels={"elasticity": "Élasticité prix", "description_short": "Produit"},
    hover_data=["StockCode", "quantity_sold", "avg_unit_price", "elasticity_profile"]
)

fig.add_vline(x=-1, line_dash="dash", annotation_text="Seuil -1")
fig.add_vline(x=0, line_dash="dash", annotation_text="Seuil 0")
fig.update_layout(height=700)
fig.show()

### Lecture métier

L'élasticité n'est pas calculable pour tous les produits, car il faut observer une variation réelle du prix dans le temps.  
Quand elle est disponible, elle donne une première indication pour le pricing dynamique.

Il faut l'utiliser comme une aide à la décision, pas comme une vérité absolue.

## 15. Recommandations pricing

On combine maintenant plusieurs signaux :

- classe ABC ;
- quadrant BCG ;
- taux de retour ;
- élasticité prix ;
- mois de pic de ventes.

L'objectif est de produire une table simple et exploitable dans le dashboard.

In [ ]:

# Mois de pic de vente par produit
product_month_revenue = (
    sales_st3.groupby(["StockCode", "month"], as_index=False)["TotalRevenue"]
    .sum()
    .rename(columns={"TotalRevenue": "month_revenue"})
)

peak_month = (
    product_month_revenue.sort_values(["StockCode", "month_revenue"], ascending=[True, False])
    .drop_duplicates("StockCode")
    .rename(columns={"month": "peak_month", "month_revenue": "peak_month_revenue"})
)

product_perf = product_perf.merge(
    peak_month[["StockCode", "peak_month", "peak_month_revenue"]],
    on="StockCode",
    how="left"
)

# Règles de recommandation simples et explicables
def pricing_recommendation(row):
    if row["return_rate_qty"] >= 0.20 and row["quantity_sold"] >= 20:
        return "Vérifier qualité / description avant promotion"

    if row["abc_class"] == "A" and row["elasticity_profile"] == "Inélastique":
        return "Tester une hausse de prix de 3 à 5 %"

    if row["bcg_quadrant"] == "Étoile":
        return "Mettre en avant et sécuriser le stock"

    if row["bcg_quadrant"] == "Dilemme":
        return "Tester visibilité ou promotion courte"

    if row["abc_class"] == "C" and row["bcg_quadrant"] == "Poids mort":
        return "Déprioriser, solder ou sortir du catalogue"

    if row["abc_class"] == "B":
        return "Maintenir et suivre la saisonnalité"

    return "Surveiller sans action urgente"

product_perf["pricing_action"] = product_perf.apply(pricing_recommendation, axis=1)

# Prix suggéré : uniquement pour les produits où on recommande un test de hausse
product_perf["suggested_price"] = np.where(
    product_perf["pricing_action"] == "Tester une hausse de prix de 3 à 5 %",
    product_perf["avg_unit_price"] * 1.05,
    product_perf["avg_unit_price"]
)

# Gain potentiel simple si hausse de 5 %, sans modélisation avancée de demande
product_perf["estimated_gain_5pct"] = np.where(
    product_perf["pricing_action"] == "Tester une hausse de prix de 3 à 5 %",
    product_perf["total_revenue"] * 0.05,
    0
)

recommendations_st3 = product_perf[
    [
        "StockCode", "description", "abc_class", "bcg_quadrant", "quantity_sold",
        "total_revenue", "avg_unit_price", "suggested_price", "elasticity",
        "elasticity_profile", "return_rate_qty", "peak_month", "pricing_action",
        "estimated_gain_5pct"
    ]
].copy()

# On affiche les recommandations les plus importantes par CA
recommendations_display = recommendations_st3.sort_values("total_revenue", ascending=False).head(30)
display(recommendations_display)

In [ ]:

action_summary = recommendations_st3.groupby("pricing_action", as_index=False).agg(
    nb_skus=("StockCode", "nunique"),
    total_revenue=("total_revenue", "sum"),
    estimated_gain_5pct=("estimated_gain_5pct", "sum")
).sort_values("total_revenue", ascending=False)

display(action_summary)

fig = px.bar(
    action_summary,
    x="pricing_action",
    y="total_revenue",
    title="Chiffre d'affaires couvert par type de recommandation pricing",
    labels={"pricing_action": "Action recommandée", "total_revenue": "Chiffre d'affaires"},
    hover_data=["nb_skus", "estimated_gain_5pct"]
)

fig.update_layout(height=600, xaxis_tickangle=-35)
fig.show()

In [ ]:

gain_total = recommendations_st3["estimated_gain_5pct"].sum()
nb_price_increase = (recommendations_st3["pricing_action"] == "Tester une hausse de prix de 3 à 5 %").sum()

message = f"""
### Lecture métier — Recommandations pricing

Le notebook identifie **{nb_price_increase} produits** pour lesquels une hausse de prix modérée peut être testée.  
Le gain potentiel théorique associé à une hausse de 5 % est d'environ **{gain_total:,.0f}**.

Cette estimation doit être utilisée avec prudence : elle donne un ordre de grandeur, mais une vraie décision pricing doit aussi tenir compte de la concurrence, de la marge, du stock et du risque de baisse de volume.
"""

display(Markdown(message))

## 16. Exports pour L8 Dashboard

On exporte les fichiers nécessaires pour construire la page Power BI / Tableau **ST3 — Catalogue Intelligence**.

Les fichiers générés dans `data/gold/` sont :

- `mart_product_perf_eda.csv` : table complète produit ;
- `top20_skus_st3.csv` : Top 20 produits ;
- `abc_summary_st3.csv` : synthèse ABC ;
- `bcg_summary_st3.csv` : synthèse BCG ;
- `seasonality_monthly_st3.csv` : saisonnalité mensuelle ;
- `seasonality_weekly_st3.csv` : saisonnalité hebdomadaire ;
- `pricing_recommendations_st3.csv` : recommandations pricing.

In [ ]:

# Colonnes principales à exporter pour le dashboard
product_export_cols = [
    "StockCode", "description", "quantity_sold", "total_revenue", "nb_orders",
    "nb_customers", "avg_unit_price", "returned_qty", "return_value",
    "return_rate_qty", "market_share_pct", "rank_revenue", "cumulative_revenue_pct",
    "abc_class", "first_period", "second_period", "growth_pct", "bcg_quadrant",
    "elasticity", "elasticity_profile", "peak_month", "pricing_action",
    "suggested_price", "estimated_gain_5pct"
]

# Certaines colonnes peuvent ne pas exister si une étape optionnelle n'a pas tourné
product_export_cols = [col for col in product_export_cols if col in product_perf.columns]

product_perf[product_export_cols].to_csv(GOLD_DIR / "mart_product_perf_eda.csv", index=False, encoding="utf-8-sig")
top20_skus.to_csv(GOLD_DIR / "top20_skus_st3.csv", index=False, encoding="utf-8-sig")
abc_summary.to_csv(GOLD_DIR / "abc_summary_st3.csv", index=False, encoding="utf-8-sig")
bcg_summary.to_csv(GOLD_DIR / "bcg_summary_st3.csv", index=False, encoding="utf-8-sig")
monthly_revenue.to_csv(GOLD_DIR / "seasonality_monthly_st3.csv", index=False, encoding="utf-8-sig")
weekly_revenue.to_csv(GOLD_DIR / "seasonality_weekly_st3.csv", index=False, encoding="utf-8-sig")
recommendations_st3.to_csv(GOLD_DIR / "pricing_recommendations_st3.csv", index=False, encoding="utf-8-sig")

print("Exports ST3 terminés dans :", GOLD_DIR)

for file in [
    "mart_product_perf_eda.csv",
    "top20_skus_st3.csv",
    "abc_summary_st3.csv",
    "bcg_summary_st3.csv",
    "seasonality_monthly_st3.csv",
    "seasonality_weekly_st3.csv",
    "pricing_recommendations_st3.csv"
]:
    print("-", file)

## 17. Synthèse finale ST3

Cette analyse permet de transformer le catalogue produit en outil de décision.

Les principaux résultats attendus sont :

1. une liste claire des produits qui génèrent le plus de chiffre d'affaires ;
2. une classification ABC pour prioriser les efforts ;
3. une matrice BCG pour distinguer les produits à pousser, maintenir ou déprioriser ;
4. une lecture saisonnière pour planifier les promotions et le stock ;
5. une estimation de l'élasticité prix pour guider les décisions de pricing ;
6. une table de recommandations directement exploitable dans le dashboard.

La prochaine étape est de construire la page **L8 — ST3 Catalogue Intelligence** dans Power BI ou Tableau à partir des exports `data/gold/`.

In [ ]:

# Résumé final dynamique
nb_products = product_perf["StockCode"].nunique()
ca_total = product_perf["total_revenue"].sum()
nb_a = int((product_perf["abc_class"] == "A").sum())
nb_stars = int((product_perf["bcg_quadrant"] == "Étoile").sum())
nb_actions = int((recommendations_st3["pricing_action"] != "Surveiller sans action urgente").sum())

summary = f"""
### Résumé dynamique

- Produits analysés : **{nb_products}**  
- Chiffre d'affaires analysé : **{ca_total:,.0f}**  
- Produits classe A : **{nb_a}**  
- Produits Étoiles : **{nb_stars}**  
- Produits avec une action recommandée : **{nb_actions}**  

Le notebook est prêt à alimenter le dashboard ST3.
"""

display(Markdown(summary))